# Scenario Playground Step 15: Ridge Explanations and Sensitivity

Objective: decompose every one of the 12 fitted Ridge predictions exactly, identify the contributions that change between the neutral baseline and user scenario, and sweep all eight shocks across representative low-, median-, and high-VIX states.

Success means every contribution reconciles numerically, sensitivity outputs are finite and deterministic, all final-test targets remain locked, and no contribution is described as causal.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import SVG, display

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "scenario_modeling").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Run this notebook from inside the InsightPulse repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scenario_modeling.src.scenario_explanations import (
    build_explanation_artifacts,
    persist_explanation_artifacts,
)
from scenario_modeling.src.scenario_inference import (
    ALL_RIDGE_TARGETS,
    CONTRACT_SHOCK_NAMES,
    build_research_inference_engine,
)

engine = build_research_inference_engine()
artifacts = build_explanation_artifacts(engine=engine)
output_paths = persist_explanation_artifacts(artifacts)
summary = artifacts["summary"]
{
    "python_executable": sys.executable,
    "completion_gate": summary["completion_gate"],
    "targets": summary["target_count"],
    "features": summary["feature_count"],
    "maximum_reconciliation_error": summary["maximum_absolute_reconciliation_error"],
    "locked_final_test": summary["locked_final_test"],
}


## Exact Ridge decomposition

For each target and row, the fitted pipeline applies its training medians and training standardization, then uses:

`prediction = intercept + sum(standardized feature value × Ridge coefficient)`

Raw values, standardized values, coefficients, and contributions remain separate so a raw zero is not mistaken for a zero model contribution.

In [ ]:
reconciliation = artifacts["prediction_reconciliation"]
display(reconciliation[[
    "target", "row_type", "model_intercept",
    "current_state_contribution", "scenario_shock_contribution",
    "release_flag_contribution", "model_prediction",
    "reconciled_prediction", "absolute_reconciliation_error",
    "reconciliation_passed",
]].round(10))
assert len(reconciliation) == 36
assert reconciliation["reconciliation_passed"].all()
assert summary["maximum_absolute_reconciliation_error"] <= 1e-9


## What changes between baseline and scenario

The current-state features, release flags, and intercept are fixed. Therefore their contribution changes must be zero; only the eight scenario-shock contribution changes may reconcile to the incremental prediction.

In [ ]:
incremental = reconciliation.loc[
    reconciliation["row_type"].eq("incremental_difference")
].copy()
display(incremental[[
    "target", "unit", "current_state_contribution",
    "scenario_shock_contribution", "release_flag_contribution",
    "model_intercept", "model_prediction",
    "absolute_reconciliation_error",
]].round(10))
assert incremental[[
    "current_state_contribution", "release_flag_contribution", "model_intercept"
]].abs().le(1e-9).all().all()


## Human-readable drivers

The deterministic templates report the strongest positive and negative standardized contributions. They preserve each target's low-confidence or weak-evidence warning and never invent financial reasoning.

In [ ]:
drivers = artifacts["top_drivers"]
driver_view = drivers.loc[
    drivers["driver_kind"].isin(["incremental_positive", "incremental_negative"])
    & drivers["rank"].eq(1),
    ["target", "driver_kind", "feature_name", "inference_role", "contribution", "explanation"],
]
display(driver_view)
assert set(driver_view["target"]) == set(ALL_RIDGE_TARGETS)


## Eight-shock sensitivity across three VIX states

Each shock is swept from its observed minimum to maximum using empirical quantiles and evenly spaced in-range values. Every other shock remains neutral. Relevant event flags are fixed for the entire sweep, and the selected market state is fixed.

In [ ]:
sensitivity = artifacts["sensitivity_summary"]
display(sensitivity[[
    "target", "contract_input", "response_directions",
    "median_prediction_span", "negligible_in_all_states",
    "unstable_across_walk_forward_folds",
    "counterintuitive_direction_flag", "all_monotonic", "all_finite",
]])
assert len(sensitivity) == 12 * 8
assert set(sensitivity["contract_input"]) == set(CONTRACT_SHOCK_NAMES)
assert sensitivity["all_monotonic"].all()
assert sensitivity["all_finite"].all()


## Plot-ready sensitivity evidence

The reusable module creates one dependency-free SVG per target, with all eight shock curves and three representative VIX states. The curves overlap where the additive Ridge response is state-invariant.

In [ ]:
for target in ALL_RIDGE_TARGETS:
    print(target)
    display(SVG(filename=output_paths[f"figure_{target}"]))
assert sum(key.startswith("figure_") for key in output_paths) == 12


## Acceptance checks

The final gate verifies exact schema/order, numeric reconciliation, sensitivity coverage, deterministic inference, neutral behavior, benchmark-only naive models, and zero locked-test access.

In [ ]:
checks = pd.Series(summary["validations"], name="passed")
display(checks.to_frame())
assert checks.all()
assert summary["completion_gate"] == "STEP_15_EXPLANATIONS_AND_SENSITIVITY_COMPLETE"
assert summary["locked_final_test"]["target_rows_parsed"] == 0
assert summary["locked_final_test"]["access_count"] == 0
print(summary["completion_gate"])


## Decision

Step 15 is accepted for Step 16 use. The explanations are exact model decompositions, not causal claims. Weak and unstable scenario behavior remains visible rather than being hidden.